# Depth Model Comparison for Corn Seed Images

Benchmark 6 monocular depth models on 200 random corn seed images.  
Each model has its own install + run cell — skip any that fail without breaking the rest.

**GPU:** A100 (40GB) — you have one.  
**Models:** Depth Pro · MoGe-2 · DepthFM · Pixel-Perfect Depth · VGGT · Depth Anything V3

## 1. Mount Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -b colab https://github.com/shurjo05/DepthComparison.git /content/DepthComparison 2>/dev/null || (cd /content/DepthComparison && git pull)
%cd /content/DepthComparison

## 2. Set Dataset Path

In [ ]:
import os

# >>> EDIT THIS to match your Google Drive path <<<
os.environ["DATASET_DIR"] = "/content/drive/MyDrive/CornSeedDetection6D/test/test"

img_dir = os.path.join(os.environ["DATASET_DIR"], "images")
lbl_dir = os.path.join(os.environ["DATASET_DIR"], "labels")
assert os.path.isdir(img_dir), f"Not found: {img_dir}"
assert os.path.isdir(lbl_dir), f"Not found: {lbl_dir}"
print(f"✓ {len([f for f in os.listdir(img_dir) if f.endswith('.jpg')])} images found")

## 3. Base Install + Select Samples

In [ ]:
!pip install -q transformers>=4.48 pillow opencv-python-headless numpy scipy trimesh
!python depth_comparison/select_samples.py
!nvidia-smi

---
## Model 1 — Depth Pro (Apple)
Metric depth. Pure HuggingFace, no extra deps.

In [ ]:
!pip install -q "transformers>=4.48"
!python depth_comparison/run_all_models.py --models depth_pro

---
## Model 2 — MoGe-2 (Microsoft)
Metric depth + surface normals + camera intrinsics.

In [ ]:
!pip install -q git+https://github.com/microsoft/MoGe.git
!python depth_comparison/run_all_models.py --models moge2

---
## Model 3 — DepthFM (CompVis)
Relative depth via flow matching. Downloads ~1.7GB checkpoint.

In [ ]:
!git clone https://github.com/CompVis/depth-fm 2>/dev/null || echo "already cloned"
!grep -v "^torch" depth-fm/requirements.txt | pip install -q -r /dev/stdin
!mkdir -p depth-fm/checkpoints
![ -f depth-fm/checkpoints/depthfm-v1.ckpt ] || wget -q --show-progress -O depth-fm/checkpoints/depthfm-v1.ckpt https://ommer-lab.com/files/depthfm/depthfm-v1.ckpt
!python depth_comparison/run_all_models.py --models depthfm

---
## Model 4 — Pixel-Perfect Depth (gangweix)
Diffusion-based relative depth. Slowest model.

In [ ]:
!git clone https://github.com/gangweix/pixel-perfect-depth 2>/dev/null || echo "already cloned"
!grep -v "^torch" pixel-perfect-depth/requirements.txt | pip install -q -r /dev/stdin
!python depth_comparison/run_all_models.py --models pixel_perfect

---
## Model 5 — VGGT (Meta)
Relative depth. Designed for multi-view but works single-image.

In [ ]:
!git clone https://github.com/facebookresearch/vggt 2>/dev/null || echo "already cloned"
!grep -v "^torch" vggt/requirements.txt | pip install -q -r /dev/stdin
!python depth_comparison/run_all_models.py --models vggt

---
## Model 6 — Depth Anything V3 (ByteDance)
Metric depth.

In [ ]:
!git clone https://github.com/ByteDance-Seed/Depth-Anything-3 2>/dev/null || echo "already cloned"
!pip install -q -e Depth-Anything-3
!python depth_comparison/run_all_models.py --models depth_anything_v3

---
## Evaluate & Compare

In [ ]:
!python depth_comparison/evaluate.py

## Results

In [ ]:
import json, glob, os
import pandas as pd
from IPython.display import display, Image as IPImage

with open("depth_comparison/results_summary.json") as f:
    results = json.load(f)

df = pd.DataFrame(results).T
display(df)

scored = {k: v for k, v in results.items() if "edge_alignment_mean" in v}
if scored:
    winner = max(scored, key=lambda k: scored[k]["edge_alignment_mean"])
    print(f"\n🏆 Best edge alignment: {winner} ({scored[winner]['edge_alignment_mean']:.4f})")

for g in sorted(glob.glob("depth_comparison/comparison_grids/*.png"))[:10]:
    print(f"\n{os.path.basename(g)}")
    display(IPImage(filename=g))